# 33 — Embeddings, Residual Stream, Position, and RoPE

**Master syllabus coverage:** V2 5.1–5.3

## Why this matters

A decoder begins by turning token IDs into continuous states and injecting order. These `[B,T,C]` states become the residual workspace used by every later block.

## Learning objectives

- Implement embedding lookup and prove its one-hot equivalence.
- Explain how repeated IDs accumulate embedding gradients.
- Construct learned and sinusoidal position representations.
- Implement RoPE on query/key head pairs and verify its invariants.
- Describe the residual stream's stable shape and additive updates.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Embeddings are learned rows selected by token ID

An embedding table $E\in\mathbb{R}^{V\times C}$ maps integer ID $i$ to row $E_i$.
Lookup is equivalent to multiplying a one-hot vector by $E$, but avoids constructing
`[B,T,V]` one-hot tensors. Repeated token IDs share one row and accumulate gradient into it.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)
B, T, V, C = 2, 5, 11, 8
token_ids = torch.tensor([[1, 4, 4, 2, 0], [1, 7, 3, 2, 0]])  # [B,T]
table = torch.randn(V, C, requires_grad=True)                   # [V,C]
looked_up = table[token_ids]                                   # [B,T,C]
one_hot = F.one_hot(token_ids, num_classes=V).float()           # [B,T,V]
via_one_hot = one_hot @ table                                  # [B,T,C]
torch.testing.assert_close(looked_up, via_one_hot)
print("embedding shape:", looked_up.shape)


## 2. Gradients collect at selected rows

A token appearing twice contributes twice to its embedding-row gradient. Tokens absent
from a batch receive zero embedding gradient for that step. Sparse lookup semantics help
interpret frequency effects and why rare token embeddings learn slowly.


In [ ]:
looked_up.sum().backward()
row_counts = torch.bincount(token_ids.flatten(), minlength=V).float()
expected_gradient = row_counts[:, None].expand(V, C)
torch.testing.assert_close(table.grad, expected_gradient)
print("token counts:", row_counts.tolist())


## 3. Without position, attention cannot know order

Token embeddings represent identity but not absolute or relative position. A classic
learned absolute scheme adds $P_t$ to every token at position $t$. Sinusoidal encodings
use fixed frequencies:

$$PE_{t,2i}=\sin(t/10000^{2i/C}),\quad
PE_{t,2i+1}=\cos(t/10000^{2i/C}).$$


In [ ]:
def sinusoidal_positions(length: int, width: int) -> torch.Tensor:
    positions = torch.arange(length, dtype=torch.float32)[:, None]
    frequencies = torch.exp(torch.arange(0, width, 2) * (-math.log(10_000.0) / width))
    encoding = torch.zeros(length, width)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding

positions = sinusoidal_positions(T, C)                  # [T,C]
residual_stream = table.detach()[token_ids] + positions # [B,T,C]
assert positions.shape == (T, C) and residual_stream.shape == (B, T, C)
print(positions.round(decimals=3))


## 4. The residual stream is a shared communication workspace

The residual stream has shape `[B,T,C]` throughout the decoder. Embeddings initialize it;
attention and MLP sublayers add updates. It is not one interpretable feature vector per
concept, nor is it copied separately for every layer. Normalization reads a transformed
view while residual additions maintain a consistent width.


In [ ]:
learned_positions = torch.nn.Embedding(T, C)
position_ids = torch.arange(T)
x = torch.nn.Embedding.from_pretrained(table.detach())(token_ids)
x = x + learned_positions(position_ids)[None, :, :]
update = torch.randn_like(x) * 0.01
x_next = x + update
assert x_next.shape == (B, T, C)
print("residual RMS before/after:", x.square().mean().sqrt().item(), x_next.square().mean().sqrt().item())


## 5. RoPE rotates queries and keys by position

Rotary position embeddings group head features into 2D pairs and rotate each pair by a
position-dependent angle. Unlike learned or sinusoidal addition to the residual stream,
RoPE is applied to queries and keys inside attention. Rotation preserves each vector norm,
while the query-key dot product encodes their relative position.

$$R_m q \cdot R_n k = q^\top R_{n-m}k$$


In [ ]:
def apply_rope(values: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
    # values: [B,H,T,D], D must be even; positions: [T]
    head_dim = values.shape[-1]
    if head_dim % 2:
        raise ValueError("RoPE head dimension must be even")
    frequencies = 1.0 / (10_000 ** (torch.arange(0, head_dim, 2, device=values.device) / head_dim))
    angles = positions[:, None] * frequencies[None, :]  # [T,D/2]
    cosine = angles.cos()[None, None, :, :]
    sine = angles.sin()[None, None, :, :]
    even, odd = values[..., 0::2], values[..., 1::2]
    rotated = torch.stack((even * cosine - odd * sine,
                           even * sine + odd * cosine), dim=-1)
    return rotated.flatten(start_dim=-2)

q = torch.randn(B, 2, T, C // 2)  # [B,H,T,D], D=4
k = torch.randn_like(q)
rope_positions = torch.arange(T)
rotated_q = apply_rope(q, rope_positions)
rotated_k = apply_rope(k, rope_positions)
torch.testing.assert_close(rotated_q.norm(dim=-1), q.norm(dim=-1))
torch.testing.assert_close(rotated_k.norm(dim=-1), k.norm(dim=-1))
print("RoPE Q/K shapes:", rotated_q.shape, rotated_k.shape)


In [ ]:
# The same q/k pair at position pairs with equal relative offset has the same dot product.
base_q = torch.randn(1, 1, 1, 4).expand(1, 1, 4, 4).clone()
base_k = torch.randn(1, 1, 1, 4).expand(1, 1, 4, 4).clone()
rq = apply_rope(base_q, torch.arange(4))
rk = apply_rope(base_k, torch.arange(4))
dot_01 = (rq[:, :, 0] * rk[:, :, 1]).sum(dim=-1)
dot_12 = (rq[:, :, 1] * rk[:, :, 2]).sum(dim=-1)
dot_23 = (rq[:, :, 2] * rk[:, :, 3]).sum(dim=-1)
torch.testing.assert_close(dot_01, dot_12, atol=1e-6, rtol=1e-5)
torch.testing.assert_close(dot_12, dot_23, atol=1e-6, rtol=1e-5)
print("equal relative offsets produce equal rotated dot products")


## Failure modes to recognize

- **Out-of-range token ID:** lookup addresses no embedding row.
- **No position signal:** reordered equal-token multisets are indistinguishable to permutation-equivariant attention.
- **Context beyond learned table:** absolute position lookup exceeds trained rows.
- **RoPE applied to values or residual states:** the architecture no longer implements rotary Q/K attention.
- **Residual width mismatch:** a sublayer update cannot join the shared stream.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Calculate embedding parameter counts for three vocabulary/width choices.
2. Verify which embedding rows receive nonzero gradients for a batch with repeated tokens.
3. Plot dot-product similarity among sinusoidal position vectors and interpret the pattern cautiously.
4. Add RoPE to the next attention notebook and compare output with and without rotation.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can produce `[B,T,C]` states from IDs and implement both additive absolute position and rotary Q/K position while explaining their different insertion points.

**Next:** 34 — Queries, keys, values, and scaled attention.
